In [1]:
import requests
from bs4 import BeautifulSoup
import time

# URL inicial da categoria
url_categoria = "https://carretaoconstrucao.com.br/pisos-e-revestimentos.html"

links_produtos = set()

# === Coleta de links ===
while url_categoria:
    print(f"Coletando página: {url_categoria}")
    response = requests.get(url_categoria)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.select("a.product-item-link"):
        href = a.get("href")
        if href and href.startswith("http"):
            links_produtos.add(href)

    next_link = soup.find("link", rel="next")
    if next_link:
        url_categoria = next_link.get("href")
        time.sleep(1)
    else:
        url_categoria = None

print(f"\nTotal de links encontrados: {len(links_produtos)}\n")

# === Extração de dados de cada produto ===
for link in sorted(links_produtos):
    try:
        print(f"Coletando dados de: {link}")
        resp = requests.get(link)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Nome
        nome = soup.find("h1", class_="page-title")
        if not nome:
            nome = soup.find("h1", {"data-ui-id": "page-title-wrapper"})
        nome = nome.get_text(strip=True) if nome else None

        # Código
        sku_div = soup.find("div", class_="product attribute sku")
        codigo = None
        if sku_div:
            value_div = sku_div.find("div", class_="value")
            if value_div:
                codigo = value_div.get_text(strip=True)

        # Preço
        preco_tag = soup.find("span", {"data-price-type": "finalPrice"})
        preco_promocional = preco_tag.get("data-price-amount") if preco_tag else None

        # Descrição
        descricao_tag = soup.find("div", class_="product attribute description")
        descricao = descricao_tag.get_text(strip=True) if descricao_tag else None

        # Imagem
        imagem_tag = soup.find("meta", {"property": "og:image"})
        imagem = imagem_tag.get("content") if imagem_tag else None

        # URL
        url_produto = soup.find("meta", {"property": "og:url"})
        url_produto = url_produto.get("content") if url_produto else link

        # Loja
        loja = "Carretão Construção"

        # Exibir dados
        dados_produto = {
            "Código": codigo,
            "Nome": nome,
            "Preço Promocional": preco_promocional,
            "Descrição": descricao,
            "Imagem": imagem,
            "URL": url_produto,
            "Loja": loja
        }

        for chave, valor in dados_produto.items():
            print(f"{chave}: {valor}")
        print("-" * 60)

    except Exception as e:
        print(f"Erro ao processar {link}: {e}")


Coletando página: https://carretaoconstrucao.com.br/pisos-e-revestimentos.html
Coletando página: https://carretaoconstrucao.com.br/pisos-e-revestimentos.html?p=2

Total de links encontrados: 17

Coletando dados de: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-30x30-preto-cozimax.html
Código: 33009
Nome: NICHO P/BANHEIRO MÁRMORE SINTÉTICO 30X30CM PRETO 89974 COZIMAX
Preço Promocional: 179.9
Descrição: Descrição do produtoO Nicho em mármore sintético 30×30 daCozimaxé uma tendência na decoração e organização de banheiros, que também pode ser utilizado em salas, cozinhas, home-offices, etc.Opcional: spot (led) para iluminação. (Modelo com luz não deve ser utilizado em áreas molhadas)Disponível nas cores:BrancoPretoBege
Imagem: https://carretaoconstrucao.com.br/media/catalog/product/cache/a0d5d6686e3b838ffaa23ceb12707527/3/3/33009.jpg.jpg
URL: https://carretaoconstrucao.com.br/nicho-marmore-sintetico-30x30-preto-cozimax.html
Loja: Carretão Construção
---------------------------